In [ ]:
import pandas as pd
import openai
import re
import time
import os
from google.colab import userdata

# ==========================================
# 1. API & MODEL CONFIGURATION
# ==========================================
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = "sk-or-v1-YOUR-KEY-HERE"

client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL_NAME = "google/gemini-3.1-flash-lite"

# ==========================================
# 2. SELECT DATASET (Uncomment ONE at a time)
# ==========================================
# INPUT_FILE, OUTPUT_FILE = "bangla_med_qa_correct.csv", "gemini_evaluated_bangla_med_qa_correct.csv"
# INPUT_FILE, OUTPUT_FILE = "wrong_answers_v1.csv", "gemini_evaluated_wrong_answers_v1.csv"
# INPUT_FILE, OUTPUT_FILE = "wrong_answers_v2.csv", "gemini_evaluated_wrong_answers_v2.csv"
INPUT_FILE, OUTPUT_FILE = "wrong_answers_v3.csv", "gemini_evaluated_wrong_answers_v3.csv"

# ==========================================
# 3. EVALUATION & PARSING FUNCTIONS
# ==========================================
def evaluate_pair(question, proposed_answer, retries=3):
    prompt = f"""You are a strict medical accuracy evaluator. Decide whether the provided model answer is correct for the question.
Only reply with a single digit: 1 or 0. No explanation, no punctuation, no extra text.
1 means the answer is factually correct and medically supported.
0 means the answer is incorrect, incomplete, hallucinated, or contradicts medical facts.

Question: {question}
Model answer: {proposed_answer}
Answer now:"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return f"ERROR: {str(e)}"

def parse_binary_score(raw_text):
    if raw_text.startswith("ERROR:"):
        return 0
    # Strip any potential tags (e.g. <think>...</think> or html tags) safely
    clean_text = re.sub(r'<.*?>', '', raw_text).strip()
    match = re.search(r'\b(0|1)\b', clean_text)
    if match:
        return int(match.group(1))
    return 1 if clean_text == "1" else 0

# ==========================================
# 4. EXECUTION LOOP
# ==========================================
def run_pipeline():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: File '{INPUT_FILE}' not found in current directory.")
        return

    print(f"Starting Evaluation on '{INPUT_FILE}' using model: {MODEL_NAME}")
    print("=" * 60)

    df = pd.read_csv(INPUT_FILE)

    raw_responses = []
    parsed_scores = []

    for idx, row in df.iterrows():
        question = row.get('question', '')
        answer = row.get('answer', '')

        raw_output = evaluate_pair(question, answer)
        score = parse_binary_score(raw_output)

        raw_responses.append(raw_output)
        parsed_scores.append(score)

        print(f"[{idx + 1}/{len(df)}] Score: {score} | Raw: '{raw_output}'")
        time.sleep(0.3)

    df['model_raw_response'] = raw_responses
    df['isCorrect'] = parsed_scores

    # Check if processing a wrong dataset or correct dataset
    is_wrong_dataset = "wrong" in INPUT_FILE.lower()

    if is_wrong_dataset:
        # Ground truth is 0 (Evaluator succeeds when isCorrect == 0)
        evaluator_correct_count = (df['isCorrect'] == 0).sum()
    else:
        # Ground truth is 1 (Evaluator succeeds when isCorrect == 1)
        evaluator_correct_count = (df['isCorrect'] == 1).sum()

    evaluator_accuracy = (evaluator_correct_count / len(df)) * 100

    df.to_csv(OUTPUT_FILE, index=False)
    print("=" * 60)
    print(f"✅ Processing Complete! Output saved to: '{OUTPUT_FILE}'")

    if is_wrong_dataset:
        print(f"   Evaluator Accuracy (Successfully flagged 0s): {evaluator_correct_count} / {len(df)} ({evaluator_accuracy:.2f}%)")
    else:
        print(f"   Evaluator Accuracy (Successfully flagged 1s): {evaluator_correct_count} / {len(df)} ({evaluator_accuracy:.2f}%)")

if __name__ == "__main__":
    run_pipeline()